# SDXL-Lightning + Euler: AnchorDraw no-op parity (smoke8)

Bước 1 của việc chuyển full AnchorDraw sang Euler Discrete. Notebook này **chưa bật semantic anchor, W03 hoặc Distillation++**, không đo metric và không lưu ảnh. Nó so sánh latent cuối của baseline SemanticDraw SDXL với cùng runtime có các điểm móc AnchorDraw identity trên 8 mẫu đầu của manifest full1073. Mỗi cặp dùng cùng model, cùng seed và cùng input; sai khác phải bằng 0.

Giao thức giữ nguyên notebook `Ours/kaggle_semanticdraw_smoke/semanticdraw_sdxl_euler_full1073_kaggle.ipynb`: SDXL base + SDXL-Lightning 4-step UNet, Euler trailing, lịch SemanticDraw `[0,4,12,25,37]` (5 lần gọi student), CFG=0, bootstrap=1, mask_std=0, 1024². Ba bản vá tương thích Kaggle của baseline cũng được áp dụng cho cả hai nhánh. Đây là kiểm tra hồi quy trong **cùng runtime**, không phải chạy lại benchmark baseline full1073 hay chứng minh ảnh trùng bit với một Kaggle session cũ khác version thư viện.


In [ ]:
# Cùng nhóm dependency generation với notebook baseline; không cài lại torch.
import subprocess, sys
packages = ['diffusers>=0.30.0', 'transformers>=4.44.0', 'accelerate', 'peft',
            'huggingface_hub', 'safetensors', 'sentencepiece', 'protobuf',
            'einops', 'pycocotools', 'pandas>=2.0']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
print('[OK] Generation dependencies installed. Restart the session if torchao was already imported.')


In [ ]:
from pathlib import Path
import os, json, importlib.util, types, zipfile
import torch

REPO_URL = 'https://github.com/GOx9-P/AnchorDraw.git'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
def is_repo(path):
    return (path / 'Baseline/semantic-draw-main/src/model/pipeline_semantic_draw_sdxl.py').is_file() and (path / 'Ours/data_manifests/coco_val2017_multidiffusion_coco_all_sdxl_1024x1024_all.jsonl').is_file()
candidates = [Path.cwd(), WORK_DIR / 'AnchorDraw', WORK_DIR / 'AnchorDraw/AnchorDraw']
REPO_ROOT = next((p.resolve() for start in candidates for p in [start, *start.parents] if is_repo(p)), None)
if REPO_ROOT is None:
    target = WORK_DIR / 'AnchorDraw'
    if not target.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(target)], check=True)
    REPO_ROOT = next((p.resolve() for p in [target, target / 'AnchorDraw'] if is_repo(p)), None)
assert REPO_ROOT is not None, 'Không tìm thấy AnchorDraw repo. Bật Internet hoặc sửa REPO_ROOT.'
assert torch.cuda.is_available(), 'Kaggle Settings -> Accelerator -> GPU.'

RUN_MANIFEST = REPO_ROOT / 'Ours/data_manifests/coco_val2017_multidiffusion_coco_all_sdxl_1024x1024_all.jsonl'
COCO_ROOT = Path(os.environ.get('COCO_ROOT', '/kaggle/working/COCO'))
OUTPUT_DIR = WORK_DIR / 'anchordraw_sdxl_euler_noop_parity_smoke8'
RUN_SAMPLES, BASE_SEED = 8, 2024
TARGET_SIZE, T_INDEX_LIST = (1024, 1024), [0, 4, 12, 25, 37]
MASK_STD, MASK_STRENGTH, PREPROCESS_MASK_COVER_ALPHA = 0.0, 1.0, 0.0
BOOTSTRAP_STEPS, GUIDANCE_SCALE = 1, 0.0
NEGATIVE_PROMPT = ''
manifest_rows = [json.loads(s) for s in RUN_MANIFEST.read_text(encoding='utf-8').splitlines() if s.strip()]
assert len(manifest_rows) == 1073 and RUN_SAMPLES <= len(manifest_rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('[OK] Repo:', REPO_ROOT, '| Manifest:', len(manifest_rows), '| Smoke:', RUN_SAMPLES)


In [ ]:
# Chỉ cần COCO annotations để giải mã mask; không tải val2017.zip hay ảnh COCO.
required = [COCO_ROOT / 'annotations/instances_val2017.json',
            COCO_ROOT / 'annotations/captions_val2017.json']
if not all(p.is_file() for p in required):
    COCO_ROOT.mkdir(parents=True, exist_ok=True)
    archive = COCO_ROOT / 'annotations_trainval2017.zip'
    if not zipfile.is_zipfile(archive):
        for url in ['https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
                    'http://images.cocodataset.org/annotations/annotations_trainval2017.zip']:
            result = subprocess.run(['wget', '-q', '--no-check-certificate', '-O', str(archive), url])
            if result.returncode == 0 and zipfile.is_zipfile(archive):
                break
        assert zipfile.is_zipfile(archive), 'Không tải được COCO annotations. Bật Internet hoặc trỏ COCO_ROOT tới dữ liệu đã có.'
    with zipfile.ZipFile(archive) as zf:
        for name in ['annotations/instances_val2017.json', 'annotations/captions_val2017.json']:
            if not (COCO_ROOT / name).is_file():
                zf.extract(name, COCO_ROOT)
assert all(p.is_file() for p in required), required
print('[OK] COCO annotations ready; val2017 images are not needed for this parity gate.')


In [ ]:
# Load hai phiên bản class từ CÙNG source: baseline đã vá Kaggle và baseline + identity hook sites.
# Không tạo model thứ hai: gọi trực tiếp __call__ của class hook trên cùng instance.
if 'torchao' in sys.modules or importlib.util.find_spec('torchao') is not None:
    raise RuntimeError('torchao vẫn import được; restart Kaggle session và Run All.')
OURS_SRC = REPO_ROOT / 'Ours/src'
BASELINE_SRC = REPO_ROOT / 'Baseline/semantic-draw-main/src'
sys.path.insert(0, str(OURS_SRC))
from data import COCORegionConfig, build_coco_region_dataloader, batch_item_to_semanticdraw_inputs
import data as _ours_data
assert Path(_ours_data.__file__).resolve().is_relative_to(OURS_SRC.resolve()), 'Wrong data module; restart Kaggle session.'
sys.path.insert(0, str(BASELINE_SRC))
from safetensors.torch import load_file as _safetensors_load_file
pipeline_path = BASELINE_SRC / 'model/pipeline_semantic_draw_sdxl.py'
source = pipeline_path.read_text(encoding='utf-8')
def replace_once(text, old, new):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f'Source drift: expected one occurrence, found {count}: {old[:90]!r}')
    return text.replace(old, new, 1)

# Giữ đúng ba patch trong notebook baseline full1073.
source = replace_once(source,
    '                latent = torch.where(count_all > 0, value / count_all, value)',
    '                denom = count_all.clamp_min(torch.finfo(count_all.dtype).tiny)\n                latent = torch.where(count_all > 0, value / denom, latent)')
source = replace_once(source,
    '                has_background = False # Regard that background does not exist.',
    '                has_background = False # Regard that background does not exist.\n                num_masks = len(prompts)\n                num_prompts = len(prompts)\n                num_nprompts = len(negative_prompts)')
source = replace_once(source,
    '        else:\n            image = latent\n\n        # Return PIL Image.',
    '        else:\n            image = latent\n\n        if output_type == "latent":\n            return image\n\n        # Return PIL Image.')

hooked_source = replace_once(source,
    '                fg_mask = fg_masks[:, i]',
    '                fg_mask = fg_masks[:, i]\n                fg_mask = self._anchor_mask_hook(fg_mask, i, t)')
hooked_source = replace_once(hooked_source,
    '                    latent_model_input = torch.cat([latent_] * 2) if do_classifier_free_guidance else latent_',
    '                    latent_ = self._anchor_pre_unet_hook(latent_, fg_mask_, i, t)\n                    latent_model_input = torch.cat([latent_] * 2) if do_classifier_free_guidance else latent_')
hooked_source = replace_once(hooked_source,
    '                    latent_ = self.scheduler_step(noise_pred, i, latent_)',
    '                    student_noisy_latent = latent_\n                    latent_ = self.scheduler_step(noise_pred, i, latent_)\n                    latent_ = self._anchor_post_euler_hook(latent_, student_noisy_latent, noise_pred, i, t)')

def load_pipeline_module(name, text):
    module = types.ModuleType(name)
    module.__file__ = str(pipeline_path)
    exec(compile(text, str(pipeline_path), 'exec'), module.__dict__)
    module.load_file = lambda filename, device=None: _safetensors_load_file(filename, device='cpu')
    return module
baseline_module = load_pipeline_module('sdxl_euler_baseline_patched', source)
hooked_module = load_pipeline_module('sdxl_euler_anchor_noop_hooks', hooked_source)
BaselineClass = baseline_module.SemanticDrawSDXLPipeline
HookedClass = hooked_module.SemanticDrawSDXLPipeline
assert source != hooked_source
print('[OK] Baseline and no-op-hook classes compiled from the same source + same compatibility patches.')


In [ ]:
# Load một student UNet duy nhất; giữ cách xử lý SDXL VAE/white bootstrap của baseline notebook.
def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        token = None
if token:
    from huggingface_hub import login
    login(token=token)

seed_everything(BASE_SEED)
smd = BaselineClass(device=torch.device('cuda:0'), dtype=torch.float16, hf_key=None,
                    has_i2t=False, t_index_list=T_INDEX_LIST, default_mask_std=MASK_STD,
                    default_mask_strength=MASK_STRENGTH,
                    default_preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA, mask_type='discrete')
if hasattr(smd.pipe, 'enable_attention_slicing'): smd.pipe.enable_attention_slicing()
if hasattr(smd.pipe, 'enable_vae_slicing'): smd.pipe.enable_vae_slicing()
if hasattr(smd.pipe, 'enable_vae_tiling'): smd.pipe.enable_vae_tiling()
def _upcast_full_vae_for_kaggle():
    smd.vae.to(dtype=torch.float32)
smd.upcast_vae = _upcast_full_vae_for_kaggle
def _safe_get_white_background(height, width):
    latent_h, latent_w = height // smd.vae_scale_factor, width // smd.vae_scale_factor
    cache = getattr(smd, '_safe_white_bootstrap_latent', None)
    if cache is None or cache.shape[-2] < latent_h or cache.shape[-1] < latent_w:
        old_dtype = smd.vae.dtype
        try:
            smd.vae.to(dtype=torch.float32)
            white_img = torch.ones(1, 3, height, width, dtype=torch.float32, device=smd.device)
            white_latent = smd.encode_imgs(white_img).to(dtype=smd.dtype)
        finally:
            smd.vae.to(dtype=old_dtype)
        if not bool(torch.isfinite(white_latent).all().item()):
            raise RuntimeError('Safe white-bootstrap latent contains NaN/Inf.')
        smd._safe_white_bootstrap_latent = white_latent
    return smd._safe_white_bootstrap_latent[..., :latent_h, :latent_w]
smd.get_white_background = _safe_get_white_background
_ = smd.get_white_background(*TARGET_SIZE)
assert type(smd.scheduler).__name__ == 'EulerDiscreteScheduler'
assert smd.scheduler.config.timestep_spacing == 'trailing'
assert smd.scheduler.config.prediction_type == 'epsilon'
assert smd.default_num_inference_steps == 4 and len(smd.timesteps) == 5
assert float(smd.gammas.abs().max()) == 0.0
assert MASK_STD == 0.0 and BOOTSTRAP_STEPS == 1 and GUIDANCE_SCALE == 0.0
print('[OK] Euler schedule:', [int(t) for t in smd.timesteps.cpu().tolist()])
print('[OK] Sigmas:', [float(s) for s in smd.sigmas.cpu().tolist()])


In [ ]:
# Cùng loader/mask/manifest với baseline; return_image=False vì parity không cần ảnh gốc.
config = COCORegionConfig(coco_root=COCO_ROOT, split='val2017',
    instances_json=COCO_ROOT / 'annotations/instances_val2017.json',
    captions_json=COCO_ROOT / 'annotations/captions_val2017.json',
    manifest_path=RUN_MANIFEST, profile='multidiffusion_coco_all', model_family='sdxl',
    target_size=TARGET_SIZE, return_image=False, cache_resized_masks=True,
    cache_dir=WORK_DIR / 'semanticdraw_mask_cache', batch_size=1, num_workers=0,
    pin_memory=False, persistent_workers=False)
loader = build_coco_region_dataloader(config, shuffle=False, drop_last=False)
assert len(loader.dataset) == 1073
print('[OK] Loader ready. First:', loader.dataset.records[0]['sample_id'])


In [ ]:
# Hai lượt trên CÙNG instance/model. Hooks identity phải thực sự được gọi và latent phải khớp bit.
from itertools import islice
audit_rows = []
for index, batch in enumerate(islice(loader, RUN_SAMPLES)):
    item = batch_item_to_semanticdraw_inputs(batch, 0)
    assert item['metadata']['sample_id'] == manifest_rows[index]['sample_id']
    seed = BASE_SEED + index
    call_args = dict(prompts=item['prompts'], negative_prompts=[NEGATIVE_PROMPT] * len(item['prompts']),
        background_prompt=item['background_prompt'], background_negative_prompt=NEGATIVE_PROMPT,
        mask_stds=MASK_STD, mask_strengths=MASK_STRENGTH, height=item['height'], width=item['width'],
        num_inference_steps=None, guidance_scale=GUIDANCE_SCALE, bootstrap_steps=BOOTSTRAP_STEPS,
        preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA, guidance_rescale=0.0,
        do_blend=False, output_type='latent')
    assert (item['height'], item['width']) == TARGET_SIZE
    seed_everything(seed)
    baseline_latent = smd(masks=item['masks'].clone().to('cuda:0', dtype=torch.float32),
                          **call_args).detach().cpu()
    counts = {'mask': 0, 'pre_unet': 0, 'post_euler': 0}
    def mask_hook(mask, step, timestep):
        counts['mask'] += 1
        return mask
    def pre_unet_hook(latent, mask, step, timestep):
        counts['pre_unet'] += 1
        return latent
    def post_euler_hook(student_next, student_noisy, noise_pred, step, timestep):
        counts['post_euler'] += 1
        assert student_next.shape == student_noisy.shape == noise_pred.shape
        return student_next
    smd._anchor_mask_hook = mask_hook
    smd._anchor_pre_unet_hook = pre_unet_hook
    smd._anchor_post_euler_hook = post_euler_hook
    seed_everything(seed)
    hooked_latent = HookedClass.__call__(smd,
        masks=item['masks'].clone().to('cuda:0', dtype=torch.float32),
        **call_args).detach().cpu()
    max_abs = float((baseline_latent.float() - hooked_latent.float()).abs().max().item())
    exact = bool(torch.equal(baseline_latent, hooked_latent))
    row = {'index': index, 'sample_id': item['metadata']['sample_id'], 'seed': seed,
           'num_foreground_regions': len(item['prompts']), 'exact': exact, 'max_abs_latent': max_abs,
           'hook_calls': dict(counts)}
    audit_rows.append(row)
    print(f"[{index+1}/{RUN_SAMPLES}] exact={exact} max_abs={max_abs:.8g} hooks={counts} {row['sample_id']}")
    assert all(value == len(T_INDEX_LIST) for value in counts.values()), row
    assert exact, f'No-op AnchorDraw path changed baseline latent: {row}'
    del baseline_latent, hooked_latent
print('[PASS] No-op AnchorDraw Euler path matches baseline latent exactly on all 8 samples.')


In [ ]:
# Chỉ lưu audit JSON; không lưu ảnh hay metric FID/IS/CLIP.
audit = {'status': 'PASS' if len(audit_rows) == RUN_SAMPLES and all(r['exact'] for r in audit_rows) else 'FAIL',
         'scope': 'same-runtime baseline vs identity-hook parity; not historical-session parity',
         'manifest': str(RUN_MANIFEST), 'samples': RUN_SAMPLES, 'seed_start': BASE_SEED,
         'model': 'SDXL base + SDXL-Lightning 4-step UNet',
         'scheduler': 'EulerDiscreteScheduler/trailing', 't_index_list': T_INDEX_LIST,
         'student_nfe': len(T_INDEX_LIST), 'guidance_scale': GUIDANCE_SCALE,
         'bootstrap_steps': BOOTSTRAP_STEPS, 'mask_std': MASK_STD,
         'anchor_enabled': False, 'w03_enabled': False, 'distillation_pp_enabled': False,
         'rows': audit_rows}
assert audit['status'] == 'PASS'
audit_path = OUTPUT_DIR / 'noop_parity_audit.json'
audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
print('[OK] Saved:', audit_path)
